# NLP Group 7 Project P2

## Imports

In [1]:
from datasets import load_dataset
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os

/Users/fannar/miniconda3/envs/nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset

In [2]:
dataset = load_dataset("MathArena/final_answer_comps", split="train")
df = dataset.to_pandas()

## Dataset Loading and Aggregation Logic
This snippet demonstrates the initial setup for creating the unified test and evaluation pool from the various competition subsets within the MathArena benchmark.

Logic: Each subset (AIME, HMMT, etc.) is loaded separately using the load_dataset function from the datasets library and immediately converted into a pandas DataFrame. The final step in the complete notebook (though not explicitly shown here) would be to concatenate all these DataFrames into a single unified test_set_df for easy looping and evaluation. The specific note about hmmt_nov_problems lacking the problem_type field is an important data cleaning consideration for the final aggregation step.

In [3]:
aime_problems = load_dataset("MathArena/aime_2025", split="train")
aime_df = aime_problems.to_pandas()

hmmt_feb_problems = load_dataset("MathArena/hmmt_feb_2025", split="train")
hmmt_feb_df = hmmt_feb_problems.to_pandas()

brumo_problems = load_dataset("MathArena/brumo_2025", split="train")
brumo_df = brumo_problems.to_pandas()

smt_problems = load_dataset("MathArena/smt_2025", split="train")
smt_df = smt_problems.to_pandas()

cmimc_problems = load_dataset("MathArena/cmimc_2025", split="train")
cmimc_df = cmimc_problems.to_pandas()


# Doesn't have the problem_type field
hmmt_nov_problems = load_dataset("MathArena/hmmt_nov_2025", split="train")
hmmt_nov_df = hmmt_nov_problems.to_pandas()



## General info and null values

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   problem_idx   139 non-null    int64 
 1   answer        139 non-null    object
 2   problem_type  130 non-null    object
 3   problem       139 non-null    object
 4   competition   139 non-null    object
 5   source        9 non-null      object
dtypes: int64(1), object(5)
memory usage: 6.6+ KB


In [5]:
print("Dataset Head:\n", df.head(), sep="", end="\n" + "="*50 + "\n")
print("Dataset Columns:\n", df.columns, sep="", end="\n" + "="*50 + "\n")
print("Dataset Shape:\n", df.shape, sep="", end="\n" + "="*50 + "\n")

Dataset Head:
   problem_idx answer                    problem_type  \
0            1     70                 [Number Theory]   
1            2    588                      [Geometry]   
2            3     16                 [Combinatorics]   
3            4    117                       [Algebra]   
4            5    279  [Combinatorics, Number Theory]   

                                             problem     competition source  
0  Find the sum of all integer bases $b>9$ for wh...  aime/aime_2025   None  
1  On $\triangle ABC$ points $A, D, E$, and $B$ l...  aime/aime_2025   None  
2  The 9 members of a baseball team went to an ic...  aime/aime_2025   None  
3  Find the number of ordered pairs $(x,y)$, wher...  aime/aime_2025   None  
4  There are $8!= 40320$ eight-digit positive int...  aime/aime_2025   None  
Dataset Columns:
Index(['problem_idx', 'answer', 'problem_type', 'problem', 'competition',
       'source'],
      dtype='object')
Dataset Shape:
(139, 6)


## Communication with the LLM model

This code block shows how the Azure AI environment is set up and how a direct API call is made to the DeepSeek-V3-03-24 model. This initialization is critical as it defines the endpoint for all subsequent Solver, Verifier, and Planner interactions.

In [6]:
load_dotenv()

ENDPOINT = "https://au803280-foundry.services.ai.azure.com/openai/v1/"
MODEL_NAME = "DeepSeek-V3-0324"
DEPLOYMENT_NAME = "DeepSeek-V3-0324"

api_key = os.getenv("API_KEY")

client = OpenAI(
    base_url=f"{ENDPOINT}",
    api_key=api_key
)

completion = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is **Paris**. It is one of the most famous and visited cities in the world, known for landmarks like the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.  \n\nWould you like information on anything specific about Paris? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


## Conditional LLM Response Formatting

The get_llm_response function is the low-level wrapper for all communication with the DeepSeek-V3-0324 model. Its most critical logic is the dynamic setting of the response_format, which is essential for ensuring the Verifier's output is reliable.

In [7]:
#Helper function
def get_llm_response(messages,  temperature=0.0):
    """Sends a request to the Azure LLM with a System Role Prompt."""
    # Configure the response format for JSON
    response_format = {"type": "text"}
    if "JSON" in messages[0] or "JSON" in messages[1]:
         response_format = {"type": "json_object"}
    
    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            temperature=temperature,
            response_format=response_format
        )
        content = response.choices[0].message.content
        return content if content is not None else ""
    except Exception as e:
        return f"LLM API Error: {e}"

## Verifier Role Definition and Output Schema (PCAF Feedback Structure)

This code block sets the system prompt ($\text{VERIFIER\_ROLE}$) for the Verifier agent and defines the strict JSON output schema it must adhere to. This structure is the fundamental communication protocol between the Verifier and the Planner.

Logic (Role): The Verifier is positioned as a Skeptical Math Auditor whose primary objective is to find flaws using a three-point checklist: Dual-Method Check, Consistency Check (text vs. code output), and Logic Trap Checks (e.g., edge cases in number theory, coordinate use in geometry).Logic (Schema): The mandatory JSON output is constrained to a specific set of $\text{error\_type}$ strings ($\text{LOGIC\_FLAW}$, $\text{VALUE\_MISMATCH}$, $\text{METHOD\_CONFLICT}$, $\text{NONE}$). These discrete, machine-readable flags are the core input for the Planner's deterministic routing logic (Section 5.3). This mechanism is designed to prevent the LLM from providing verbose, unstructured text that is difficult to parse and act upon.

In [8]:
VERIFIER_ROLE = (
    "You are a Skeptical Math Auditor. Your goal is to catch LOGICAL ERRORS that standard code checks miss.\n\n"

    "### AUDIT CHECKLIST\n"
    "1. **The 'Double-Check' Rule**: Did the Solver use **two different approaches** (e.g., Simulation vs. Formula) to confirm the answer?\n"
    "   - If they only used one method (e.g., just a formula) and it looks complex/risky, mark as **Risky** (but pass if confident).\n"
    "   - If they used two methods and they **CONFLICT**, REJECT immediately.\n\n"

    "2. **The 'Consistency' Rule**: Does the [Code Output] match the text? (Crucial).\n\n"

    "3. **Logic Trap Checks**:\n"
    "   - **Combinatorics**: Did they brute force it? If they used a formula like `nCr`, did they verify it for small N?\n"
    "   - **Geometry**: Did they use coordinates? Visual assumptions (e.g., 'it looks like a square') are grounds for REJECTION.\n"
    "   - **Number Theory**: Did they handle edge cases (e.g., 0, 1, negatives)?\n\n"

    "### OUTPUT FORMAT\n"
    "Return ONLY this JSON:\n"
    "{\n"
    "  \"valid\": <boolean>,\n"
    "  \"error_type\": <\"LOGIC_FLAW\" | \"VALUE_MISMATCH\" | \"METHOD_CONFLICT\" | \"NONE\">,\n"
    "  \"critique\": <string: If rejected, explain WHY the logic is suspect. If passed, confirming the dual-verification.>\n"
    "}"
)


## Tool Implementation: PersistentSolverSandbox

This class is the foundation of the "Code-First Mandate" in the PCAF framework, ensuring reliable, stateful, and safe code execution by the Solver.

Logic (Persistence and State): The key architectural feature is maintaining the code execution state. The self.globals dictionary holds all defined variables and pre-loaded libraries (like $\text{sympy}$ and $\text{numpy}$) across sequential calls to run_code. This allows the Solver to execute multi-step logic (e.g., define a variable in one $\text{```python```}$ block and access it in the next) via the $\text{exec(code\_str, self.globals)}$ command.

Logic (Robustness and Safety): The run_code method is wrapped by func_timeout(timeout_seconds, ...) to impose a strict 20-second time limit. This prevents infinite loops or overly complex calculations from halting the entire system. All execution outputs and errors are reliably captured using $\text{contextlib.redirect\_stdout}$ and returned to the LLM with the $\text{RUNTIME ERROR}$ tag, enabling the Solver to self-debug its code errors in the next turn.

In [9]:
#Helper function

import sys
import io
import contextlib
import textwrap
# Make sure you have run: !pip install func_timeout
from func_timeout import func_timeout, FunctionTimedOut

class PersistentSolverSandbox:
    def __init__(self):
        self.globals = {
            "__builtins__": __builtins__,
            "print": print
        }
        self._exec_setup()
        
    def _exec_setup(self):
        setup_code = """
import math
import sympy
import numpy as np
import itertools
import sys

# 1. Safe Math Imports
from math import sqrt, sin, cos, tan, log, exp, pi, e, factorial, acos, asin, atan, radians, degrees

# 2. Robust SymPy Imports
from sympy import symbols, solve, nsolve, Eq, simplify, expand, factor, Rational, isprime, N
from sympy import Reals, S 

# 3. Config
sys.set_int_max_str_digits(0)
"""
        try:
            exec(setup_code, self.globals)
        except Exception as e:
            print(f"Sandbox Setup Error: {e}")

    def run_code(self, code_str, timeout_seconds=20):
        # 1. Auto-Fix Indentation
        try:
            code_str = textwrap.dedent(code_str)
        except:
            pass

        # 2. Auto-Clean Non-ASCII Characters (Fixes the '≈' crash)
        code_str = code_str.replace("≈", "=").replace("≠", "!=").replace("≤", "<=").replace("≥", ">=")

        output_buffer = io.StringIO()
        error_buffer = io.StringIO()
        
        def _run_captured():
            with contextlib.redirect_stdout(output_buffer), contextlib.redirect_stderr(error_buffer):
                exec(code_str, self.globals)
        
        try:
            func_timeout(timeout_seconds, _run_captured)
            
            stdout_val = output_buffer.getvalue()
            stderr_val = error_buffer.getvalue()
            
            # 3. Combine output
            full_output = stdout_val
            if stderr_val:
                full_output += f"\n[WARNINGS/ERRORS]:\n{stderr_val}"
            
            # 4. Check for "Empty" success (The Silent Killer)
            if not full_output.strip():
                return "[SYSTEM]: Code executed but produced NO OUTPUT. Did you forget to print(result)? Variables are NOT returned automatically."
            
            # 5. Check for Empty Lists specifically
            if full_output.strip() == "[]":
                return "[]\n[SYSTEM]: Your code returned an empty list. Check your variable ranges or equations."

            if len(full_output) > 2500: 
                return full_output[:2500] + f"\n... [OUTPUT TRUNCATED. Total: {len(full_output)} chars]"
            
            return full_output
            
        except FunctionTimedOut:
            return "RUNTIME ERROR: Code execution exceeded time limit (20s). Loop likely infinite."
        except IndentationError:
            return "RUNTIME ERROR: IndentationError. Ensure code is properly formatted."
        except Exception as e:
            return f"RUNTIME ERROR: {str(e)}"

# --- Unit Test the Sandbox ---
test_code = """
import sympy
x = sympy.symbols('x')
sol = sympy.solve(x**2 - 4, x)
print(sol)
"""
env = PersistentSolverSandbox()
print(f"Sandbox Test Output: {env.run_code(test_code)}")

persist_test_code = """
print(sol)
"""
print(f"Persistency test output: {env.run_code(persist_test_code)}")

empty_test_code = """
print([])
"""
print(f"Empty list test output: {env.run_code(empty_test_code)}")

Sandbox Test Output: [-2, 2]

Persistency test output: [-2, 2]

Empty list test output: []
[SYSTEM]: Your code returned an empty list. Check your variable ranges or equations.


## Execution Kernel

The run_solver_with_tools function is the operational core of the Solver agent. It manages the conversational turns within a single attempt, dynamically integrating the LLM's thought process with the results of the external PersistentSolverSandbox.

Logic (Tool Integration Loop): The function iteratively checks the LLM's response for Python code blocks using a regular expression.
-  Code Detection: If one or more code blocks are found, the conversation loop pauses the LLM's generation.
-  Execution and Feedback: Each code block is executed sequentially by the sandbox_instance.run_code(). The combined outputs including any RUNTIME_ERRORs are formatted and fed back to the LLM via a new user message tagged as $\text{OBSERVATION (Code Output)}$.
-  Trace Logging: The full conversation and the execution output are meticulously compiled into the $\text{full\_trace}$ variable, with the output marked by the $\text{[Tool Output]}$ tag. This trace is the complete evidentiary record passed to the Verifier for auditing.

Logic (Completion): The loop continues for a maximum of $\text{max\_turns}$ (3). It terminates successfully if the LLM's response contains a final answer marker (\\\boxed or $\text{Final Answer}$), indicating the Solver has reached its conclusion. If no final answer is reached after all turns, the final partial response is returned for verification.

In [10]:
#Helper function
import re
def run_solver_with_tools(system_prompt, user_problem, sandbox_instance, temperature=0.7, max_turns=3):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_problem}
    ]
    full_trace = ""
    
    for turn in range(max_turns):
        response = get_llm_response(messages, temperature=temperature)
        full_trace += f"\n\n--- Step {turn+1} ---\n{response}"
        
        # Find ALL code blocks
        code_blocks = re.findall(r"```python\n(.*?)```", response, re.DOTALL)
        
        if code_blocks:
            print(f"    [Tool Use] Detected {len(code_blocks)} code blocks.")
            
            combined_output = ""
            for i, code_str in enumerate(code_blocks):
                print(f"    [Block {i+1}] Executing...")
                out = sandbox_instance.run_code(code_str)
                # FORMATTING CHANGE HERE: Use a consistent tag for the trace
                combined_output += f"[Block {i+1} Output]:\n{out}\n\n"
            
            print(f"    [Tool Output] {combined_output.strip()[:100]}...")
            
            # Feed back to LLM (Model sees 'OBSERVATION' which is standard convention)
            messages.append({"role": "assistant", "content": response})
            messages.append({
                "role": "user", 
                "content": f"OBSERVATION (Code Output):\n{combined_output}\n\nContinue reasoning."
            })
            
            # Feed to Trace (We use [Tool Output] so the Verifier Regex can find it)
            # --- FIX IS HERE ---
            full_trace += f"\n\n[Tool Output]\n{combined_output}"
            # -------------------
            
        else:
            if "\\boxed" in response or "Final Answer" in response:
                return response, full_trace
            
            messages.append({"role": "assistant", "content": response})
            if turn == max_turns - 1:
                break
                
    return response, full_trace

## Few-Shot Examples for Solver Self-Correction

The FEW\_SHOT\_CORRECTION\_EXAMPLE string is a structured prompt component used to instruct the Solver on how to react to specific rejection signals from the Verifier. This is vital for transferring the logic of the PCAF loop into the LLM's behavioral model, ensuring corrections are targeted.
The examples illustrate the three primary failure modes defined by the Verifier and the corresponding corrective action expected from the Solver:

In [11]:
#Not helper function but different file
FEW_SHOT_CORRECTION_EXAMPLE = r"""
## REFERENCE EXAMPLE: Self-Correction from a LOGIC_FLAW

Problem: Find the number of integers between 1 and 50 (inclusive) that are divisible by 2 OR divisible by 3.

---
### TURN 0: Initial Solve (Flawed Logic)

Reasoning (Initial Attempt):
1. I need to find the count of numbers divisible by 2 and add the count of numbers divisible by 3.
2. Initial Flaw: I will initially forget to subtract the double-counted numbers (divisible by 6).

```python
count_by_2 = 50 // 2
count_by_3 = 50 // 3
initial_flawed_count = count_by_2 + count_by_3
print(initial_flawed_count)```

OBSERVATION (Code Output): 41
Final Answer (Initial Attempt): 41
Verifier Verdict: INVALID (LOGIC_FLAW) Planner Instruction: The Verifier identified a LOGIC_FLAW. 
Feedback: The count is incorrect. The Solver failed to apply the Principle of Inclusion-Exclusion (PIE). 
Numbers divisible by both 2 and 3 (i.e., by 6) have been counted twice.
MANDATORY ACTION: RE-SOLVE. Correct the logic by subtracting the overlap (divisible by 6).

### TURN 1: Solver Correction (Corrected Logic)
Reasoning (Correction):
The Planner noted a LOGIC_FLAW due to missing the Principle of Inclusion-Exclusion (PIE).
The correct count is: (Divisible by 2) + (Divisible by 3) - (Divisible by LCM(2, 3) = 6).
I will correct the Python logic to subtract the overlap.

```python
# Count (Divisible by 2)
count_by_2 = 50 // 2 
# Count (Divisible by 3)
count_by_3 = 50 // 3 
# Count (Divisible by 6) - This is the overlap to subtract
count_by_6 = 50 // 6 
# Apply PIE
final_correct_count = count_by_2 + count_by_3 - count_by_6
print(final_correct_count)```

OBSERVATION (Code Output): 33
Reasoning: The corrected count using PIE is 33. 
Final Answer: 33

## EXAMPLE 2: Self-Correction from a METHOD_CONFLICT
Problem: Find the probability that an integer selected randomly from $[1, 100]$ is prime, rounded to 4 decimal places.

### TURN 0: Initial Solve (Analytical vs Simulation Conflict)
Reasoning (Initial Attempt):
Method 1 (Analytical): Count the primes (25) and divide by 100.
Method 2 (Simulation): Run a Monte Carlo simulation for verification.

### Method 1: Analytical
```python
prob_analytical = 25 / 100
print("Method 1:", prob_analytical)

# Method 2: Simulation (Monte Carlo)
# ... code to run 10000 trials ...
# For this example, assume it output 0.2488
prob_sim = 0.2488
print("Method 2:", prob_sim)```

OBSERVATION (Code Output - Example): Method 1: 0.25 Method 2: 0.2488
Final Answer (Initial Attempt): 0.2500 Verifier Verdict: INVALID (METHOD_CONFLICT) Planner Instruction: The Verifier detected a conflict between the analytical value (0.25) and the simulation value (0.2488).
The conflict is due to the simulation being an approximation.
MANDATORY ACTION: RE-SOLVE. Since Method 1 (Analytical) is exact, trust Method 1 and ensure the final output is rounded correctly.

### TURN 1: Solver Correction (Resolution)Reasoning (Correction):The conflict arose because the simulation is stochastic. 
The analytical method is deterministic and correct ($0.25$).
I will re-state the final answer using the deterministic result, ensuring the required 4 decimal places.

```python
# Final confirmation of the analytical, deterministic result
final_result = 0.25
print(final_result)```

OBSERVATION (Code Output): 0.25
Reasoning: The analytical result is $0.25$. Rounded to four decimal places, the answer is $0.2500$.
Final Answer: 0.2500

## EXAMPLE 3: Self-Correction from a VALUE_MISMATCH
Problem: Calculate the value of $5^{5}$ and report the result modulo 100.

### TURN 0: Initial Solve (Calculation Correct, Final Answer Flawed)
Reasoning (Initial Attempt):
1. The required calculation is $5^5$. I will calculate the value directly using Python.
2. Flaw: I will calculate the value correctly ($3125$) but forget the final modulo 100 operation in the Final Answer.

```python
calculated_value = 5**5
print(calculated_value)

OBSERVATION (Code Output): 3125
Final Answer (Initial Attempt): 3125 
Missed the modulo 100 operation Verifier Verdict: INVALID (VALUE_MISMATCH) 
Planner Instruction: The Verifier found that the final answer (3125) does not match the ground truth (25). 
The code output is correct but the required final transformation was missed. MANDATORY ACTION: RE-SOLVE.
Apply the required transformation (modulo 100) to the calculated value and ensure the Final Answer reflects this.

### TURN 1: Solver Correction (Applying Final Transformation)
Reasoning (Correction):
The Planner noted a VALUE_MISMATCH because I missed the modulo 100 transformation. 
The calculated value is 3125. The correct final answer must be $3125 \pmod{100}$.
I will correct the Python logic and output the final required value.

```python
calculated_value = 5**5
final_result = calculated_value % 100
print(final_result)```

OBSERVATION (Code Output): 25 
Reasoning: The final value after applying the modulo 100 transformation is 25.
Final Answer: 25"""


## Solver Agent System Prompt Generation

This function dynamically generates the SOLVER\_ROLE system prompt, dictating the Solver agent's behavior and enforcing the framework's core requirement for verifiable, code-grounded solutions. We implemented a Dual-Verification Protocol: The central mechanism is the "ANALYTICAL vs NAIVE" CHECK. This mandates that the Solver must use two fundamentally different, cross-validating methods:
- Method 1 (Analytical): Symbolic and efficient algorithms (leveraging libraries like sympy).
- Method 2 (Naive Brute-Force): Simple, unoptimized simulation or counting loops.
This dual-approach ensures that if the Analytical method contains a subtle logical error, the Brute-Force "Dumb Check" will likely produce a conflicting answer, which the Verifier will then catch as a $\text{METHOD\_CONFLICT}$.

When a retry is requested ($\text{is\_retry=True}$), the function injects a strong ATTENTION warning. 

In [12]:
def get_solver_prompt(few_shot_examples=None, is_retry=False):
    base_prompt = (
        "You are a Rigorous Mathematical Solver. To guarantee accuracy, you must solve the problem using **TWO STRICTLY DIFFERENT METHODS**.\n\n"
        
        "### PROTOCOL: THE 'ANALYTICAL vs NAIVE' CHECK\n"
        "1. **Method 1: Analytical/Symbolic**: Use `sympy`, algebra, or efficient algorithms to solve the problem mathematically.\n"
        "2. **Method 2: Naive Brute-Force (The 'Dumb' Check)**: \n"
        "   - Write a **simple, unoptimized Python loop** that iterates through all possibilities.\n"
        "   - **DO NOT** use formulas or clever shortcuts in Method 2. Just count or simulate step-by-step.\n"
        "   - *Example*: If counting divisors, Method 1 uses prime factors; Method 2 iterates `for i in range(1, n+1): if n%i==0...`\n"
        "   - *Example*: If probability, Method 1 uses combinatorics; Method 2 runs a `random.choice` simulation 100,000 times.\n"
        "3. **Comparison**: Check if Method 1 == Method 2.\n"
        "   - If they MATCH: Print the result.\n"
        "   - If they DIFFER: Trust Method 2 (the brute force) or debug Method 1. Do not hallucinate a match.\n\n"
        
        "### OUTPUT RULES\n"
        "- Write all code in ```python ... ``` blocks.\n"
        "- You MUST explicitly `print()` the result from BOTH methods.\n"
        "- Final Answer must be the value confirmed by BOTH methods.\n"
    )
    
    if few_shot_examples:
        base_prompt += f"\n## REFERENCE EXAMPLES\n{few_shot_examples}\n"
    
    if is_retry:
        base_prompt += (
            "\n\n**ATTENTION:** Your previous attempt was REJECTED. "
            "You likely hardcoded the answer or your code output didn't match your text. "
            "Wipe your memory of the 'known' answer and re-derive it from scratch using code."
        )
        
    return base_prompt

# FOR BASELINE (Current State):
SOLVER_ROLE = get_solver_prompt(few_shot_examples=FEW_SHOT_CORRECTION_EXAMPLE)

# FOR FUTURE PCAF (Future State):
# few_shots = "... content from MathInstruct ..."
# SOLVER_ROLE_PCAF = get_solver_prompt(few_shot_examples=few_shots)

## Planner Agent: Deterministic Correction Routing

The construct\_planner\_feedback function is the implementation of the Planner agent. It performs the vital role of converting the Verifier's discrete JSON error type into a highly specific, prescriptive, and mandatory instruction for the Solver. This deterministic routing is the core mechanism that breaks the Solver's failure loop.

Logic (Deterministic Routing): The function uses a series of if/elif statements based on the $\text{error\_type}$ extracted from the Verifier's JSON output: 
- EXECUTION_FAILURE / RUNTIME ERROR: The instruction focuses solely on debugging the code syntax or outputting a value, delaying the math logic correction until the code runs.
- VALUE_MISMATCH (The Stubborn Fix): This triggers a Protocol Reset and Code-Grounding Enforcement. The Solver is told to IGNORE its previous text reasoning and RE-EXECUTE its code, accepting the code's output as the truth. This prevents the Solver from clinging to a numerically wrong answer.
- HARDCODING_SUSPICION (The Anti-Cheat Fix): The Solver is issued a METHODOLOGY REJECTION and forced to DERIVE the answer using a Python script (e.g., a loop for counting, $\text{sympy.solve}$ for equations), preventing it from outputting "magic numbers" or relying on recalled answers.
- CONCEPTUAL_FLAW / Geometry (The Strategy Shift): If a geometry-related term is found in the problem text, the Planner issues a STRATEGY SHIFT, commanding the Solver to abandon its previous flawed approach and switch to Coordinate Geometry. This forces a known-reliable, algorithmic method ($\text{Shoelace Formula}$, etc.), which is easier to verify.

In [13]:
def construct_planner_feedback(verifier_json, problem_text, history_text=""):
    """
    Routes specific error types to 'Loop Breaking' instructions.
    """
    error_cat = verifier_json.get("error_type", "UNKNOWN") # Note: Key changed to error_type to match new prompt
    critique = verifier_json.get("critique", "")

    # 1. HANDLE RUNTIME ERRORS (Keep existing logic)
    last_turn_trace = history_text.split("--- Correction")[-1] if "--- Correction" in history_text else history_text
    if "RUNTIME ERROR" in last_turn_trace or error_cat == "EXECUTION_FAILURE":
        return (
            "The previous code attempt failed with a RUNTIME ERROR or produced NO OUTPUT. "
            "**MANDATORY ACTION:** Fix the syntax. Do not change the math logic yet; just make the code run. "
            "Print the final result explicitly."
        )

    # 2. HANDLE "VALUE_MISMATCH" (The "Stubborn Solver" Fix)
    # This fixes the issue in Problem 23 where Text=279 but Code=610
    if error_cat == "VALUE_MISMATCH":
        return (
            f"**CRITICAL CONFLICT:** The Verifier found a mismatch. {critique}. "
            "Your text claims one answer, but your Python code calculated a different one. "
            "**PROTOCOL:** \n"
            "1. IGNORE your previous text intuition. It was wrong.\n"
            "2. **RE-EXECUTE the code** to confirm the calculated value is stable.\n"
            "   Do NOT copy your previous reasoning. Write a COMPLETELY NEW derivation text that matches the new code."
            "3. If the code runs correctly, accept its output as the truth.\n"
            "**WARNING:** Do NOT just state the number from the feedback. You MUST run the code again to prove it."
        )

    # 3. HANDLE "HARDCODING_SUSPICION" (The "Anti-Cheat" Fix)
    # This prevents the "I recall..." or manual counting issues
    if error_cat == "HARDCODING_SUSPICION":
        return (
            "**METHODOLOGY REJECTION:** The Verifier flagged your answer as a 'Magic Number' or 'Manual Count'. "
            "**MANDATORY ACTION:** You must Delete your 'recalled' answer. Write a Python script to DERIVE the answer from scratch. "
            "If counting items, write a loop. If solving equations, use `sympy.solve`."
        )

    # 4. HANDLE "CONCEPTUAL_FLAW" (Geometry/Logic Routing)
    if error_cat == "CONCEPTUAL_FLAW":
        is_geometry = any(k in problem_text.lower() for k in ["geometry", "triangle", "polygon", "area"])
        if is_geometry:
             return (
                 "**STRATEGY SHIFT:** Your geometric reasoning was flagged as flawed. "
                 "**MANDATORY ACTION:** Switch to Coordinate Geometry. Place a vertex at (0,0), define all other points as (x,y) coordinates, and use the Shoelace Formula or Distance Formula in Python."
             )
        else:
            return (
                "**LOGIC ERROR:** Your mathematical approach is flawed. "
                "Step back. List the constraints again. Try a different method (e.g., if you used Algebra, try Brute Force with Python)."
            )

    # Fallback
    return f"The Verifier failed the solution. Reason: {critique}. Please fix the code to match the reasoning."

## Final Answer Extraction Logic

The extract\_answer function is a multi-strategy heuristic designed for robustly identifying and cleaning the final, scoreable answer from the Solver's output trace. This is crucial for automation because LLM output formats can be inconsistent.
- Strategy 1: $\text{\boxed{}}$ (Highest Priority): The function first scans for the $\text{\boxed{}}$ LaTeX command. The logic is specifically implemented to handle nested braces by using a counter ($\text{brace\_count}$), ensuring it accurately extracts the content even if the box contains complex LaTeX or internal grouping.
- Strategy 2: $\text{Final Answer}$ Marker: If no box is found, it searches for explicit textual markers like "Final Answer:" or "answer is:".Robust Token Extraction: Critically, it then applies a sub-strategy to extract only the numerical/mathematical token from the matched line. This prevents common errors where the final answer is $\text{18.897}$ but the Solver appends text like "cards)" or "is the result," which would corrupt the scoring.
- Strategy 3: Last Math Token (Fallback): As a final, low-confidence heuristic, the function extracts all mathematical tokens (integers, decimals, fractions, $\sqrt{}$, ${\pi}$) from the entire text and returns the last one found. This captures answers where the Solver fails to use any explicit marker.

In [14]:
#Helper function
def extract_answer(text):
    """
    Robust extraction that handles nested \boxed{}, cleans up text artifacts,
    and prioritizes explicit answer markers.
    """
    if not isinstance(text, str): return None
    
    # --- Strategy 1: Boxed Content (Recursive for nested braces) ---
    start_idx = text.find(r'\boxed{')
    if start_idx != -1:
        content_start = start_idx + 7 # Length of "\boxed{"
        brace_count = 1
        i = content_start
        while i < len(text) and brace_count > 0:
            if text[i] == '{':
                brace_count += 1
            elif text[i] == '}':
                brace_count -= 1
            i += 1
        
        if brace_count == 0:
            return text[content_start : i-1].strip()

    # --- Strategy 2: Explicit "Final Answer" text ---
    final_match = re.search(r'(?:Final Answer|answer is)[:\s]*([^\n]+)(?:\n|$)', text, re.IGNORECASE)
    if final_match:
        candidate = final_match.group(1).strip()
        
        # --- FIX START: Clean the candidate string ---
        # Instead of returning the whole line (which might contain "cards)"), 
        # we try to extract the specific math token from it.
        
        # 1. Normalize fractions
        candidate_norm = re.sub(r'\\(?:d)?frac\{([^{}]+)\}\{([^{}]+)\}', r'\1/\2', candidate)
        
        # 2. Extract Math Tokens (same pattern as Strategy 3)
        token_pattern = r'(?:-?\d+(?:,\d{3})*(?:\.\d+)?(?:/\d+)?|\\sqrt\{[^{}]+\}|\\pi)'
        tokens = re.findall(token_pattern, candidate_norm)
        
        if tokens:
            return tokens[-1].strip() # Returns "18.897" from "18.897 cards)"
            
        # Fallback: Just strip common punctuation if no math token is found
        return candidate.rstrip(".:,;!)]}")
        # --- FIX END ---

    # --- Strategy 3: Last Math Token (Heuristic Fallback) ---
    text_norm = re.sub(r'\\(?:d)?frac\{([^{}]+)\}\{([^{}]+)\}', r'\1/\2', text)
    token_pattern = r'(?:-?\d+(?:,\d{3})*(?:\.\d+)?(?:/\d+)?|\\sqrt\{[^{}]+\}|\\pi)'
    candidates = re.findall(token_pattern, text_norm)
    
    if candidates:
        return candidates[-1].strip()

    return None

## Evaluation Utility (Semantic Scoring):

The $\text{check\_corectness}$ function provides the final, robust layer of evaluation by ensuring that mathematically equivalent answers are scored correctly, regardless of their formatting (e.g., $\text{1/2}$ vs. $\text{0.5}$).
Logic (Numerical Equivalence): The core mechanism is a helper function that converts complex input strings (both the prediction and the ground truth) into Python-executable expressions.
- LaTeX Conversion: It uses regular expressions to convert raw LaTeX fractions (${\frac\{a\}\{b\}}$) into Python division syntax ${(a)/(b)}$, and maps math constants (like ${\sqrt{}}$ and ${\pi}$) to the Python $\text{math}$ library functions.
- Implied Multiplication Fix: Critically, it inserts a multiplication operator where implied multiplication exists (e.g., changing ${2\pi}$ to ${2*\pi}$), preventing common execution errors during evaluation.
Final Check: It uses the resulting numerical values to perform a check based on floating-point tolerance ($\text{1e-3}$), guaranteeing semantic correctness for non-integer answers.

In [15]:
import math
#Helper function
def check_correctness(prediction, ground_truth):
    """
    Semantically checks if the prediction matches the ground truth.
    Handles Commas, LaTeX formatting (dfrac/frac), and Implied Multiplication.
    """
    if prediction is None or ground_truth is None:
        return False
        
    pred_str = str(prediction).strip()
    gt_str = str(ground_truth).strip()

    # --- 1. String Normalization Check ---
    def clean_string(s):
        s = s.replace(" ", "").replace(",", "")
        s = s.replace(r"\boxed", "").replace(r"\text", "")
        s = s.replace(r"\dfrac", r"\frac") # Normalize fraction types
        s = s.replace(r"\(", "").replace(r"\)", "")
        return s
        
    p_clean = clean_string(pred_str)
    g_clean = clean_string(gt_str)

    # Direct match (now handles dfrac vs frac)
    if p_clean == g_clean:
        return True
        
    # Substring match (Safety net for extraction failures)
    if g_clean in p_clean and len(p_clean) < len(g_clean) * 4:
        return True

    # --- 2. Numerical/Symbolic Evaluation ---
    def safe_eval(s):
        try:
            # Pre-processing for eval
            s = s.replace(",", "") 
            s = s.replace("^", "**")
            s = s.replace(r"\dfrac", r"\frac")
            # Handle LaTeX fractions: \frac{a}{b} -> (a)/(b)
            s = re.sub(r'\\frac\{([^{}]+)\}\{([^{}]+)\}', r'(\1)/(\2)', s)
            s = s.replace(r"\sqrt", "sqrt").replace(r"\pi", "pi")
            s = s.replace("{", "(").replace("}", ")")
            s = s.replace("\\", "") 

            # Fix Implied Multiplication: "2sqrt" -> "2*sqrt", "2pi" -> "2*pi", "2(" -> "2*("
            # We insert a * if a digit is followed immediately by a letter or open paren
            s = re.sub(r'(\d)\s*([a-zA-Z(])', r'\1*\2', s)
            
            # Restricted eval environment
            safe_env = {"math": math, "sqrt": math.sqrt, "pi": math.pi, "abs": abs}
            return float(eval(s, {"__builtins__": {}}, safe_env))
        except Exception:
            return None

    val_pred = safe_eval(prediction)
    val_gt = safe_eval(ground_truth)

    if val_pred is not None and val_gt is not None:
        if abs(val_pred - val_gt) < 1e-3:
            return True

    return False

## Data Bridge:



The $\text{extract\_json\_from\_response}$ function is a crucial data bridge that ensures the Verifier's LLM output (raw text critique) is reliably converted into a structured input for the Planner's deterministic logic. 
- Multi-Strategy Parsing: The function is built for extreme robustness by using sequential parsing attempts to handle different malformed outputs from the LLM:
    - It cleans up Markdown wrappers (e.g., $\text{```json}$).
    - It attempts standard JSON loading after fixing issues like trailing commas.
    - It falls back to $\text{ast.literal\_eval}$ for Python dictionary syntax (single quotes, $\text{True/False}$ booleans).
- Guaranteed Data Validation (Core Feature): The $\text{validate\_and\_default}$ helper is the most important part of this logic. After successful parsing, it checks if essential keys (like $\text{valid}$ and $\text{error\_type}$) are present. If they are missing, it injects safe default values ($\text{False}$ and $\text{NONE}$) to prevent the downstream Planner's deterministic routing logic from crashing due to unexpected schema holes.

In [16]:
import json
import re
import ast
#Helper Function
def extract_json_from_response(text):
    """
    Robustly extracts JSON or Python-Dictionary-like objects from a string.
    Handles Markdown, trailing commas, single quotes, and Python booleans.
    Injects default keys if missing to prevent downstream errors.
    """
    if text is None: return None

   # --- NEW: Helper to ensure required keys exist ---
    def validate_and_default(data):
        if not isinstance(data, dict):
            return data
            
        # 1. Standardize Key Names (Map 'error_category' -> 'error_type')
        if "error_category" in data and "error_type" not in data:
            data["error_type"] = data.pop("error_category")
            
        # 2. Inject Defaults if missing
        if "valid" not in data: 
            data["valid"] = False
        if "error_type" not in data: 
            data["error_type"] = "NONE" # Default to avoid crash
        if "critique" not in data: 
            # Check for old key 'critique_summary' just in case
            if "critique_summary" in data:
                data["critique"] = data.pop("critique_summary")
            else:
                data["critique"] = "The answer is incorrect but no specific critique was provided."
        return data
    # -------------------------------------------------

    # 1. Strip Markdown Code Blocks
    text = re.sub(r"```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```python\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)
    text = text.strip()

    # 2. Strategy A: Regex Search for the outermost brace structure
    match = re.search(r"(\{.*\})", text, re.DOTALL)
    if match:
        candidate = match.group(1)
    else:
        candidate = text

    # 3. Strategy B: Try Standard JSON Parsing
    try:
        return validate_and_default(json.loads(candidate))
    except json.JSONDecodeError:
        pass 

    # 4. Strategy C: Clean Common Syntax Errors
    candidate_clean = re.sub(r",\s*\}", "}", candidate)
    try:
        return validate_and_default(json.loads(candidate_clean, strict=False))
    except json.JSONDecodeError:
        pass

    # 5. Strategy D: Python Literal Eval
    try:
        candidate_python = candidate_clean.replace("true", "True").replace("false", "False").replace("null", "None")
        return validate_and_default(ast.literal_eval(candidate_python))
    except (ValueError, SyntaxError):
        pass

    # 6. Failure
    return None

## PCAF Orchestration Kernel

The $\text{run\_pcaf\_on\_problem}$ function implements the central control flow loop: Solve $\rightarrow$ Verify $\rightarrow$ Plan $\rightarrow$ (Re-)Solve. It manages the multi-agent state to enable iterative self-correction, limited by a hard constraint.
Core Logic (The Constrained Loop): 
- The system is designed to execute a maximum of four attempts (one initial attempt + $\text{max\_retries}=3$ corrections). The loop controls the sequencing of the three agents (Solver, Verifier, Planner) until a valid solution is found or the limit is reached.
- Correction Enforcement: Each correction step ensures targeted debugging by passing the full history of failures and a prescriptive $\text{planner\_instruction}$ back to the Solver. If $\text{is\_retry=True}$, the Solver is commanded to perform a Protocol Reset ("Wipe your memory...") to prevent it from repeating the past mistake.
- State Management: The loop persists the single, stateful $\text{PersistentSolverSandbox}$ across all four attempts, allowing complex multi-step code workflows to continue even after a correction/retry.

In [17]:
def run_pcaf_on_problem(problem_text, max_retries=3, temperature=0.7):
    all_tries = []
    problem_sandbox = PersistentSolverSandbox()
    
    # 1. Initial Attempt
    current_solution, trace = run_solver_with_tools(
        get_solver_prompt(is_retry=False), 
        problem_text,
        problem_sandbox,
        temperature=temperature
    )
    
    # Initialize loop variables
    # We treat the initial attempt as "Attempt 0"
    structured_history = [f"=== ATTEMPT 1 (ORIGINAL) ===\n{trace}"]
    
    # We loop up to max_retries + 1 because we want to Verify the INITIAL attempt too.
    # The loop condition controls how many CORRECTIONS we allow.
    for attempt_idx in range(max_retries + 1):
        
        # --- A. VERIFY CURRENT SOLUTION ---
        print(f"--- Verifying Attempt {attempt_idx + 1} ---")
        
        # Extract evidence from the CURRENT trace
        code_outputs = re.findall(r"\[Tool Output\]\s*(.*?)(?=\n\n--- Step|\Z)", trace, re.DOTALL)
        if not code_outputs:
            evidence_str = "NO CODE EXECUTED."
        else:
            evidence_str = ""
            for idx, out in enumerate(code_outputs):
                evidence_str += f"=== EXECUTION {idx+1} ===\n{out.strip()}\n\n"

        verifier_input = (
            f"Problem: {problem_text}\n\n"
            f"=== EVIDENCE (CODE OUTPUTS) ===\n{evidence_str}\n\n"
            f"=== SOLVER'S REASONING ===\n{trace}"
        )
        
        verifier_raw = get_llm_response([
            {"role": "system", "content": VERIFIER_ROLE},
            {"role": "user", "content": verifier_input}
        ])
        verifier_json = extract_json_from_response(verifier_raw)
        
        # Log this attempt
        current_try = {
            "turn": attempt_idx,
            "solution_text": current_solution,
            "full_trace": trace,
            "verifier_json": verifier_json,
            "planner_instruction": None
        }
        all_tries.append(current_try)
        
        # --- B. CHECK SUCCESS ---
        if verifier_json and verifier_json.get("valid", False):
            print(f"Verifier Verdict: VALID (Attempts: {attempt_idx + 1})")
            return current_solution, all_tries
        
        print(f"Verifier Verdict: INVALID ({verifier_json.get('error_type')})")
        
        # --- C. STOP IF OUT OF RETRIES ---
        if attempt_idx == max_retries:
            print("Max retries reached. Stopping.")
            break
            
        # --- D. PLAN & CORRECT (If we have retries left) ---
        planner_instruction = construct_planner_feedback(verifier_json, problem_text, trace)
        all_tries[-1]["planner_instruction"] = planner_instruction # Log instruction
        
        print(f"--- Generating Correction {attempt_idx + 1} ---")
        history_text = "\n\n".join(structured_history)
        
        solver_input_context = (
            f"ORIGINAL PROBLEM: {problem_text}\n\n"
            f"--- HISTORY OF FAILURES ---\n{history_text}\n\n"
            f"--- FEEDBACK ---\n"
            f"VERIFIER: {verifier_json.get('critique', 'Invalid')}\n"
            f"INSTRUCTION: {planner_instruction}\n"
        )
        
        current_solution, new_trace = run_solver_with_tools(
            get_solver_prompt(is_retry=True),
            solver_input_context,
            problem_sandbox,
            temperature=temperature
        )
        
        # Update trace for the next loop iteration
        trace = new_trace 
        structured_history.append(f"=== ATTEMPT {attempt_idx + 2} (CORRECTION) ===\n{new_trace}")

    return current_solution, all_tries

## Evaluation Protocol:

This function applies the MathArena evaluation protocol to the PCAF system, testing not just correctness, but reliability across multiple independent runs.
- Protocol (Multiple Independent Runs): The core mechanism is a double loop. For every problem, the entire $\text{run\_pcaf\_on\_problem}$ cycle (the full Solve-Verify-Plan loop) is executed four times ($\text{attempts\_per\_problem=4}$). Each of these four runs is a separate, fresh, and independent evaluation of the PCAF.
- Final Metric (Averaged Accuracy): The score for a single problem is not binary ($\text{0}$ or $\text{1}$), but the average accuracy across the four runs. For instance, if the PCAF solves the problem correctly in 3 out of 4 independent runs, the problem's score is $\text{0.75}$. This protocol accounts for the inherent stochasticity of LLMs (due to $\text{temperature=0.7}$) and provides a much more robust measure of system performance.
- Data Logging: It diligently logs the results from all four runs, including the answers, the scores, and the full histories ($\text{run\_histories}$), providing the necessary data to analyze system stability and self-correction behavior under variance.

In [ ]:
def run_matharena_pcaf_benchmark(dataframe, num_samples=None, attempts_per_problem=4, temperature=0.7):
    """
    Evaluates the PCAF System using MathArena protocol:
    - 4 independent PCAF runs per problem.
    - Score is average accuracy of the 4 final PCAF outputs.
    """
    subset = dataframe.head(num_samples).copy() if num_samples else dataframe.copy()
    results = []

    print(f"=======================================================")
    print(f"STARTING MATHARENA PCAF EVALUATION")
    print(f"Problems: {len(subset)} | PCAF Runs/Prob: {attempts_per_problem} | Solver Temp: {temperature}")
    print(f"=======================================================")

    for index, row in subset.iterrows():
        problem_id = row['problem_idx']
        ground_truth = str(row['answer'])
        problem_text = row['problem']
        
        print(f"\n--- Problem ID: {problem_id} ---")
        
        run_scores = []
        run_answers = []
        run_turns = []
        run_histories = []
        
        for i in range(attempts_per_problem):
            # Run the FULL PCAF LOOP (Solver+Verifier+Planner)
            # This counts as ONE attempt
            final_sol, history = run_pcaf_on_problem(
                problem_text, 
                max_retries=3, 
                temperature=temperature
            )
            
            # Extract final answer from the result of the PCAF loop
            extracted_val = extract_answer(final_sol)
            is_correct = check_correctness(extracted_val, ground_truth)
            
            score = 1 if is_correct else 0
            run_scores.append(score)
            run_answers.append(extracted_val)
            run_turns.append(len(history))
            run_histories.append(history)
            
            # Log result
            mark = "True" if is_correct else "False"
            # We track how many internal turns PCAF took (e.g., did it self-correct?)
            print(f"   PCAF Run {i+1}: {mark} (Ans: {extracted_val}) [Internal Turns: {len(history)}]")
            
        # Calc Stats
        avg_score = sum(run_scores) / attempts_per_problem
        print(f"   >> Avg PCAF Accuracy: {avg_score:.2f}")
        
        results.append({
            "problem_idx": problem_id,
            "ground_truth": ground_truth,
            "pcaf_answers": run_answers,
            "pcaf_scores": run_scores,
            "pcaf_turns": run_turns,
            "final_score": avg_score,
            # Serialize history to JSON so it saves cleanly to CSV
            "pcaf_full_histories": [json.dumps(h) for h in run_histories]
        })
        
    results_df = pd.DataFrame(results)
    print(f"\nGlobal PCAF Accuracy: {results_df['final_score'].mean() * 100:.2f}%")
    return results_df

# --- EXECUTION --- 
competitions = (aime_df, hmmt_feb_df)
for index, comp_df in enumerate(competitions):
    pcaf_matharena_results = run_matharena_pcaf_benchmark(comp_df, num_samples=1)
    pcaf_matharena_results.to_csv(f"PCAF_Final_Result_{index}.csv", index=False) 
    print(f"\nDetailed results saved to PCAF_Final_Result_{index}.csv")

STARTING MATHARENA PCAF EVALUATION
Problems: 1 | PCAF Runs/Prob: 4 | Solver Temp: 0.7

--- Problem ID: 1 ---
    [Tool Use] Detected 1 code blocks.
    [Block 1] Executing...
    [Tool Output] [Block 1 Output]:
Brute-force method result: [21, 49]
Sum of valid bases: 70...


KeyboardInterrupt: 